In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [2]:
!pip -q install datasets pandas

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import os
import random
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = "/content/drive/MyDrive/ICT4403_MT_Project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Dataset directory:", DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project directory: /content/drive/MyDrive/ICT4403_MT_Project
Dataset directory: /content/drive/MyDrive/ICT4403_MT_Project/data


In [4]:
dataset = load_dataset("Helsinki-NLP/opus-100", "en-es")

print(dataset)
print("\nAvailable splits:", list(dataset.keys()))

for split in dataset.keys():
    print(f"{split}: {len(dataset[split]):,} examples")

print("\nExample:")
print(dataset["train"][0])

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-es/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  237kB            

en-es/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-es/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 99.6MB            

en-es/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

en-es/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})

Available splits: ['test', 'train', 'validation']
test: 2,000 examples
train: 1,000,000 examples
validation: 2,000 examples

Example:
{'translation': {'en': "It was the asbestos in here, that's what did it!", 'es': 'Fueron los asbestos aquí. ¡Eso es lo que ocurrió!'}}


In [5]:
def clean_split(split_name):
    df = dataset[split_name].to_pandas()

    # OPUS-100 stores each pair in a translation dictionary
    df = pd.DataFrame({
        "en": df["translation"].apply(lambda x: str(x["en"]).strip()),
        "es": df["translation"].apply(lambda x: str(x["es"]).strip())
    })

    # Remove empty pairs and exact duplicates
    df = df[(df["en"] != "") & (df["es"] != "")]
    df = df.drop_duplicates(subset=["en", "es"])

    return df.reset_index(drop=True)


train_all = clean_split("train")
val_all = clean_split("validation")
test_all = clean_split("test")

print("Cleaned training pool:", len(train_all))
print("Cleaned validation pool:", len(val_all))
print("Cleaned test pool:", len(test_all))

Cleaned training pool: 929682
Cleaned validation pool: 1998
Cleaned test pool: 1999


In [6]:
TRAIN_SIZE = 20_000
VAL_SIZE = 1_000
TEST_SIZE = 1_000

if len(train_all) < TRAIN_SIZE:
    raise ValueError("Not enough training examples for the selected subset.")

if len(val_all) < VAL_SIZE or len(test_all) < TEST_SIZE:
    raise ValueError("Not enough official validation/test examples.")

train_df = train_all.sample(
    n=TRAIN_SIZE, random_state=SEED
).reset_index(drop=True)

val_df = val_all.sample(
    n=VAL_SIZE, random_state=SEED
).reset_index(drop=True)

test_df = test_all.sample(
    n=TEST_SIZE, random_state=SEED
).reset_index(drop=True)

# Save the exact same splits for every model notebook
train_df.to_csv(os.path.join(DATA_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(DATA_DIR, "validation.csv"), index=False)
test_df.to_csv(os.path.join(DATA_DIR, "test.csv"), index=False)

print("Saved all three splits.")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

display(train_df.head())

Saved all three splits.
Train: (20000, 2)
Validation: (1000, 2)
Test: (1000, 2)


,en,es
0,Are you sure it's a kill chip and not a tracker?,¿Estás seguro que es un chip asesino y no un r...
1,The representative of Bolivia thanked the Secr...,El representante de Bolivia agradeció al Secre...
2,It is however acknowledged that it is appropri...,"No obstante, se ha admitido la conveniencia de..."
3,"I know, Brick.","Lo sé, Brick."
4,Don't fuck with the dosage. Wow.,No juegues con la medicación.


In [7]:
import json

metadata = {
    "dataset": "OPUS-100",
    "huggingface_id": "Helsinki-NLP/opus-100",
    "configuration": "en-es",
    "source_language": "English",
    "target_language": "Spanish",
    "seed": SEED,
    "train_size": len(train_df),
    "validation_size": len(val_df),
    "test_size": len(test_df),
    "preprocessing": [
        "Trim leading and trailing whitespace",
        "Remove empty sentence pairs",
        "Remove exact duplicate pairs within each split",
        "Sample training data from the official training split",
        "Sample validation and test data from their respective official splits"
    ]
}

with open(os.path.join(DATA_DIR, "metadata.json"), "w") as f:
    json.dump(metadata, f, indent=2)

print(json.dumps(metadata, indent=2))

{
  "dataset": "OPUS-100",
  "huggingface_id": "Helsinki-NLP/opus-100",
  "configuration": "en-es",
  "source_language": "English",
  "target_language": "Spanish",
  "seed": 42,
  "train_size": 20000,
  "validation_size": 1000,
  "test_size": 1000,
  "preprocessing": [
    "Trim leading and trailing whitespace",
    "Remove empty sentence pairs",
    "Remove exact duplicate pairs within each split",
    "Sample training data from the official training split",
    "Sample validation and test data from their respective official splits"
  ]
}


In [8]:
import os
import pandas as pd
import json

print("Dataset files:")
for filename in ["train.csv", "validation.csv", "test.csv", "metadata.json"]:
    path = os.path.join(DATA_DIR, filename)
    print(f"{filename}: {'FOUND' if os.path.exists(path) else 'MISSING'}")

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(DATA_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print("\nDataset sizes:")
print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nMissing values:")
print(pd.concat({
    "train": train_df.isna().sum(),
    "validation": val_df.isna().sum(),
    "test": test_df.isna().sum()
}, axis=1))

print("\nSample test pairs:")
display(test_df.head(5))

with open(os.path.join(DATA_DIR, "metadata.json")) as f:
    print("\nMetadata:")
    print(json.dumps(json.load(f), indent=2))

Dataset files:
train.csv: FOUND
validation.csv: FOUND
test.csv: FOUND
metadata.json: FOUND

Dataset sizes:
Train: (20000, 2)
Validation: (1000, 2)
Test: (1000, 2)

Missing values:
    train  validation  test
en      0           0     0
es      0           0     0

Sample test pairs:


,en,es
0,We humans will inevitably end up controlling o...,Nosotros humanos terminaremos inevitablemente ...
1,If she had not rented quarter in this building...,Si no hubiera rentado un cuarto en este edific...
2,I know what the lies were.,Sé cuáles eran las mentiras.
3,- Here's your buddy for a party...,! - A eso a venido la chavala a la fiesta...
4,This is not possible for a man of flesh and bl...,Esto no es posible para un hombre de carne y h...



Metadata:
{
  "dataset": "OPUS-100",
  "huggingface_id": "Helsinki-NLP/opus-100",
  "configuration": "en-es",
  "source_language": "English",
  "target_language": "Spanish",
  "seed": 42,
  "train_size": 20000,
  "validation_size": 1000,
  "test_size": 1000,
  "preprocessing": [
    "Trim leading and trailing whitespace",
    "Remove empty sentence pairs",
    "Remove exact duplicate pairs within each split",
    "Sample training data from the official training split",
    "Sample validation and test data from their respective official splits"
  ]
}


In [9]:
import os
import shutil
from google.colab import files

# Project paths
PROJECT_DIR = "/content/drive/MyDrive/ICT4403_MT_Project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

# Create a temporary folder for the backup
BACKUP_DIR = "/content/ICT4403_MT_Backup"
os.makedirs(BACKUP_DIR, exist_ok=True)

# Copy the dataset files
for filename in [
    "train.csv",
    "validation.csv",
    "test.csv",
    "metadata.json"
]:
    source = os.path.join(DATA_DIR, filename)

    if os.path.exists(source):
        shutil.copy2(source, BACKUP_DIR)
        print("Added:", filename)
    else:
        print("WARNING — file not found:", filename)

# Create the ZIP archive
ZIP_PATH = "/content/ICT4403_MT_Dataset_Backup"
shutil.make_archive(ZIP_PATH, "zip", BACKUP_DIR)

print("\nZIP created:", ZIP_PATH + ".zip")

# Download to your computer
files.download(ZIP_PATH + ".zip")

Added: train.csv
Added: validation.csv
Added: test.csv
Added: metadata.json

ZIP created: /content/ICT4403_MT_Dataset_Backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>